In [1]:
# define server
# define client
# data should be financial bank information 
# probably want to work with horizontally partitioned data??
# going to partition personal bank loan data into 4 clients randomly, though it all comes from the same database                                                                                                                                                                                                           `1`

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

np.random.seed(69)

In [15]:
# divide bank data into 4 clients randomly
# make 4 dataframes, 1 for each client
df = pd.read_csv("data.csv")
print(df.head())

# only thing that needs preprocessing is the ccavg
df['CCAvg']=df['CCAvg'].str.replace("/",".")
# somebody using this data for a classification project converted this to per year so i will do that too
df['CCAvg']=df['CCAvg'].astype(float)*12


def split_data(df, num_clients=4):
    df = df.sample(frac=1).reset_index(drop=True)
    client_data = []
    for i in range(num_clients):
        client_data.append(df.iloc[i::num_clients, :].reset_index(drop=True))
    return client_data
print(split_data(df,4))


   ID  Age  Experience  Income  ZIP Code  Family CCAvg  Education  Mortgage  \
0   1   25           1      49     91107       4  1/60          1         0   
1   2   45          19      34     90089       3  1/50          1         0   
2   3   39          15      11     94720       1  1/00          1         0   
3   4   35           9     100     94112       1  2/70          2         0   
4   5   35           8      45     91330       4  1/00          2         0   

   Personal Loan  Securities Account  CD Account  Online  CreditCard  
0              0                   1           0       0           0  
1              0                   1           0       0           0  
2              0                   0           0       0           0  
3              0                   0           0       0           0  
4              0                   0           0       0           1  
[        ID  Age  Experience  Income  ZIP Code  Family  CCAvg  Education  \
0      110   43        

In [ ]:
# make client class that holds the data for each client and also trains a local model on the data
# the idea is sending an SGD update to the server every time a client trains a model on their local data
# so the client has to be able to do an SGD update, send it to the server, and receive the updated model from the server

class Client:
    def __init__(self, client_id, data):
        self.client_id = client_id
        self.data = data
        self.theta = None
        self.learning_rate = 0.01
        self.threshold = 1.0
        self.X_train, self.X_test, self.y_train, self.y_test = self.prepare_data()
    def prepare_data(self):
        X = self.data.drop('Personal Loan', axis=1)
        y = self.data['Personal Loan']
        return train_test_split(X, y, test_size=0.2)
    def update_model(self, theta):
        # will only be called by the server
        self.theta = theta
    def SGD(self):
        # needs to only be one update, not a full training of the model
        if self.theta is None:
            self.theta = np.zeros(self.X_train.shape[1])
        else:
            # perform one step of SGD
            idx = np.random.randint(0, self.X_train.shape[0])
            x_i = self.X_train.iloc[idx].values
            y_i = self.y_train.iloc[idx]
            prediction = 1 / (1 + np.exp(-np.dot(self.theta, x_i)))
            gradient = (prediction - y_i) * x_i
            self.theta -= self.learning_rate * gradient    
    def clip(self):
        # clip the gradient
        if self.theta is not None:
            norm = np.linalg.norm(self.theta)
            if norm > self.threshold:
                self.theta = self.theta * (self.threshold / norm)
    def differential_privacy(self, epsilon=1.0):
        # adding noise to the gradient
        if self.theta is not None:
            noise = np.random.laplace(0, 1/epsilon, size=self.theta.shape)
            self.theta += noise
    def send_to_server(self):
        self.SGD()
        self.clip()
        self.differential_privacy()
        return self.theta
    def evaluate(self):
        predictions = 1 / (1 + np.exp(-np.dot(self.X_test, self.theta)))
        predictions = np.round(predictions)
        accuracy = np.mean(predictions == self.y_test)
        return accuracy

In [25]:
# initalize 4 clients 
clients = []
for i in range(4):
    clients.append(Client(i, split_data(df,4)[i]))

# do 2 updates to make sure everything is working
for _ in range(2):
    for client in clients:
        print(client.send_to_server())

# im getting values, don't know if they are correct or not but i'll move on for now

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[-1.15000e+00 -2.40000e-01 -1.20000e-01 -3.55000e-01 -4.65585e+02
 -1.00000e-02 -1.02000e-01 -5.00000e-03 -7.25000e-01  0.00000e+00
  0.00000e+00  0.00000e+00 -5.00000e-03]
[-1.61150e+01 -2.45000e-01 -1.15000e-01 -4.05000e-01 -4.65535e+02
 -1.00000e-02 -4.80000e-02 -1.50000e-02  0.00000e+00  0.00000e+00
  0.00000e+00  0.00000e+00  0.00000e+00]
[-1.1555e+01 -1.6000e-01 -3.0000e-02 -1.6000e-01 -4.6403e+02 -1.0000e-02
 -1.8000e-02 -5.0000e-03  0.0000e+00  0.0000e+00  0.0000e+00  0.0000e+00
  0.0000e+00]
[9.40000e-01 2.30000e-01 1.05000e-01 7.95000e-01 4.71525e+02 1.50000e-02
 1.14000e-01 1.50000e-02 1.57500e+00 0.00000e+00 0.00000e+00 5.00000e-03
 0.00000e+00]


In [26]:
class Server:
    def __init__(self, clients):
        self.clients = clients
        # need to initialize theta.... not sure how i should start the variables
        # i'll try all 0s 
        self.theta = np.zeros(clients[0].X_train.shape[1])
    def aggregate(self):
        # average the theta values from the clients
        theta_sum = np.zeros(self.theta.shape)
        for client in self.clients:
            theta_sum += client.send_to_server()
        self.theta = theta_sum / len(self.clients)
        # send the updated theta back to the clients
        for client in self.clients:
            client.update_model(self.theta)

In [27]:
server = Server(clients)
for i in range(10):
    server.aggregate()
    print(f"iteration {i} : {server.theta}")

iteration 0 : [-1.5207500e+01 -2.0875000e-01 -7.0000000e-02 -1.0375000e-01
 -4.6493375e+02 -1.1250000e-02 -7.3500000e-02 -1.0000000e-02
  2.1250000e-01  0.0000000e+00  0.0000000e+00  1.2500000e-03
 -1.2500000e-03]
iteration 1 : [-1.6525000e+00 -1.6375000e-01 -8.5000000e-02  1.5212500e+00
 -2.7142875e+02  1.1250000e-02  4.6950000e-01 -2.2500000e-02
  9.9500000e-01  0.0000000e+00  0.0000000e+00  1.1250000e-02
 -1.2500000e-03]
iteration 2 : [ 5.862500e+00 -8.750000e-03  5.000000e-03  1.891250e+00 -3.615125e+01
  1.875000e-02  6.795000e-01 -1.750000e-02  9.950000e-01  0.000000e+00
  0.000000e+00  1.375000e-02 -1.250000e-03]
iteration 3 : [ 2.6050000e+01  1.5875000e-01  1.1250000e-01  2.4937500e+00
  2.2056875e+02  1.3750000e-02  7.6350000e-01 -5.0000000e-03
  9.9500000e-01  0.0000000e+00  1.0000000e-02  1.6250000e-02
  1.2500000e-03]
iteration 4 : [ 1.5710000e+01 -4.4125000e-01 -2.2750000e-01  2.2037500e+00
 -7.3916125e+02 -6.2500000e-03  7.2750000e-01 -1.5000000e-02
  9.9500000e-01  0.000

C:\Users\mjc18\AppData\Local\Temp\ipykernel_32344\3804841495.py:28: RuntimeWarning: overflow encountered in exp
  prediction = 1 / (1 + np.exp(-np.dot(self.theta, x_i)))


In [28]:
# evaluate the model on each client
for client in clients:
    accuracy = client.evaluate()
    print(f"client {client.client_id} accuracy: {accuracy}")

client 0 accuracy: 0.88
client 1 accuracy: 0.92
client 2 accuracy: 0.916
client 3 accuracy: 0.908


C:\Users\mjc18\AppData\Local\Temp\ipykernel_32344\3804841495.py:35: RuntimeWarning: overflow encountered in exp
  predictions = 1 / (1 + np.exp(-np.dot(self.X_test, self.theta)))


In [ ]:
# now i am going to practice using different privacy techniques
# gradient clipping
# differential privacy
# privacy accounting
# secure aggregation
# encrypted communication
# mpc

In [ ]:
# gradient clipping is a standard method for preventing exploding gradients in deep learning models. But we can also use it in federated learning to stop the clients from sending gradients that are too large
# this way, one client doesn't dominate the information sent to the serer.
# i put this back into the client class, so look for clip() to see it

# differential privacy is a technique that adds noise to the gradients before sending them to the server
# this way the server cannot learn a lot about each individual contribution of the client
# there is an accuracy tradeoff though
# once again i added it to the client class, so look for differential_privacy() to see it

# privacy accounting is measuring how much privacy loss has been spent over training
# you start with a budget of privacy loss and each time you send an update to the server you spend some of that budget
# this is useful for comparing different models and seeing which one is more privacy preserving 
